# Capítulo 8: Cuando los Árboles No Suficienten (Deep Learning)

> *"Deep Learning es como un misil guiado: perfecto para ciertos objetivos, un desperdicio absoluto para otros."*

**Objetivo:** Entender cuándo usar Deep Learning (y cuándo NO usarlo). Comparar MLP vs XGBoost en tabulares, explorar CNNs para imágenes, LSTMs para series temporales, y Transformers para texto. Reflexionar sobre los sesgos en modelos de DL.

**La regla de oro:** *"Si tus datos son tabulares, no uses DL. Si son imágenes, texto o audio, aquí está el manual."*

## Celda 1: Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn para ML tradicional
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# XGBoost para comparación
try:
    import xgboost as xgb
    print('XGBoost disponible:', xgb.__version__)
except ImportError:
    print('XGBoost no instalado. Instalar con: pip install xgboost')

# TensorFlow/Keras para Deep Learning
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    print('TensorFlow disponible:', tf.__version__)
except ImportError:
    print('TensorFlow no instalado. Instalar con: pip install tensorflow')

# Transformers para NLP
try:
    from transformers import pipeline
    print('Transformers disponible')
except ImportError:
    print('Transformers no instalado. Instalar con: pip install transformers')

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 100)
np.random.seed(42)
print('\nLibrerías cargadas correctamente.')

## Celda 2: Ejemplo MLP para tabulares (y por qué NO deberías usarlo)

Vamos a demostrar por qué Deep Learning NO es la mejor opción para datos tabulares. Usaremos el mismo enfoque que los capítulos anteriores.

In [ ]:
# Simular datos tabulares similares a HR Analytics
np.random.seed(42)
n_samples = 2000

data = {
    'Age': np.random.randint(18, 60, n_samples),
    'MonthlyIncome': np.random.normal(5000, 2000, n_samples).clip(1000, 20000),
    'YearsAtCompany': np.random.randint(0, 30, n_samples),
    'JobSatisfaction': np.random.randint(1, 5, n_samples),
    'WorkLifeBalance': np.random.randint(1, 5, n_samples),
    'PerformanceRating': np.random.randint(1, 5, n_samples),
    'DistanceFromHome': np.random.randint(1, 30, n_samples),
    'NumCompaniesWorked': np.random.randint(0, 10, n_samples),
}

# Crear target con relaciones no lineales
prob = (
    0.3 * (data['JobSatisfaction'] < 3).astype(int) +
    0.25 * (data['WorkLifeBalance'] < 2).astype(int) +
    0.2 * (data['MonthlyIncome'] < 3000).astype(int) +
    0.15 * (data['YearsAtCompany'] > 10).astype(int) +
    0.1 * np.random.random(n_samples)
)
data['Attrition'] = (prob > 0.5).astype(int)

df = pd.DataFrame(data)
print(f'Dimensiones: {df.shape}')
print(f'\nDistribución de Attrition:')
print(df['Attrition'].value_counts())
df.head(10)

In [ ]:
# Preparar datos
X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar para MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')
print(f'\nFeatures: {list(X.columns)}')

In [ ]:
# ============================================
# MODELO 1: MLP (Deep Learning para tabulares)
# ============================================
print('=' * 60)
print('MODELO 1: MLP (Multi-Layer Perceptron)')
print('=' * 60)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3 capas ocultas
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

print(f'\nArquitectura: {mlp.hidden_layer_sizes}')
print(f'Parámetros totales: {sum(p.size for p in mlp.coefs_) + sum(p.size for p in mlp.intercepts_):,}')
print(f'\nAccuracy: {accuracy_score(y_test, y_pred_mlp):.4f}')
print(f'\nReporte de clasificación:')
print(classification_report(y_test, y_pred_mlp))

## Celda 3: Comparación con XGBoost (el tabular gana)

Comparemos el MLP con XGBoost y Random Forest. **Spoiler: los árboles ganan.**

In [ ]:
# ============================================
# MODELO 2: Random Forest
# ============================================
print('=' * 60)
print('MODELO 2: Random Forest')
print('=' * 60)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'\nReporte de clasificación:')
print(classification_report(y_test, y_pred_rf))

# ============================================
# MODELO 3: XGBoost
# ============================================
print('\n' + '=' * 60)
print('MODELO 3: XGBoost')
print('=' * 60)

try:
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    
    print(f'\nAccuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
    print(f'\nReporte de clasificación:')
    print(classification_report(y_test, y_pred_xgb))
except:
    print('XGBoost no disponible')
    y_pred_xgb = None

In [ ]:
# ============================================
# COMPARACIÓN FINAL
# ============================================
print('=' * 60)
print('COMPARACIÓN: MLP vs Random Forest vs XGBoost')
print('=' * 60)

results = {
    'MLP (Deep Learning)': accuracy_score(y_test, y_pred_mlp),
    'Random Forest': accuracy_score(y_test, y_pred_rf),
}
if y_pred_xgb is not None:
    results['XGBoost'] = accuracy_score(y_test, y_pred_xgb)

print('\nResultados:')
for model, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {model}: {acc:.4f}')

winner = max(results, key=results.get)
print(f'\n🏆 Ganador: {winner}')

# Feature Importance de Random Forest
print('\n\nFeature Importance (Random Forest):')
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)
for feat, imp in importance.items():
    bar = '█' * int(imp * 50)
    print(f'  {feat:20s} {imp:.4f} {bar}')

print('\n' + '=' * 60)
print('CONCLUSIÓN: Para datos tabulares, los árboles ganan.')
print('MLP es más lento, más difícil de tuneear, y menos explicitable.')
print('=' * 60)

## Celda 4: Ejemplo CNN simple para imágenes

Aquí es donde DL SÍ brilla: imágenes. Veamos una CNN simple para clasificación de imágenes.

**Metáfora:** Las CNNs son como "ojos de halcón" para imágenes: detectan bordes, formas, y objetos progresivamente.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ============================================
# CNN para MNIST (ejemplo clásico)
# ============================================
print('=' * 60)
print('CNN para clasificación de imágenes (MNIST)')
print('=' * 60)

# Cargar dataset MNIST
(X_train_img, y_train_img), (X_test_img, y_test_img) = tf.keras.datasets.mnist.load_data()

# Normalizar y reshape
X_train_img = X_train_img.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_img = X_test_img.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f'Train images: {X_train_img.shape}')
print(f'Test images: {X_test_img.shape}')
print(f'Clases: {np.unique(y_train_img)}')

# Visualizar algunas imágenes
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_img[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {y_train_img[i]}')
    ax.axis('off')
plt.suptitle('Ejemplos de imágenes MNIST', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Arquitectura CNN
# ============================================
model_cnn = models.Sequential([
    # Primera capa convolucional
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Segunda capa convolucional
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Tercera capa convolucional
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Capas densas
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')  # 10 dígitos
])

model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Arquitectura CNN:')
model_cnn.summary()

# Entrenar (solo 5 epochs para demostración)
print('\n\nEntrenando CNN...')
history = model_cnn.fit(
    X_train_img, y_train_img,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Evaluar
test_loss, test_acc = model_cnn.evaluate(X_test_img, y_test_img, verbose=0)
print(f'\nAccuracy en test: {test_acc:.4f}')

In [ ]:
# Visualizar entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Entrenamiento de CNN', fontsize=14)
plt.tight_layout()
plt.show()

print('\n' + '=' * 60)
print('CONCLUSIÓN: CNNs son ideales para imágenes.')
print('Detectan patrones espaciales que los árboles no pueden.')
print('=' * 60)

## Celda 5: Ejemplo LSTM para series temporales

Las LSTMs son ideales para secuencias donde el orden importa.

**Metáfora:** Un LSTM es como un lector que lleva un cuaderno de notas: recuerda información importante del pasado para tomar mejores decisiones en el presente.

In [ ]:
# ============================================
# LSTM para predicción de series temporales
# ============================================
print('=' * 60)
print('LSTM para series temporales')
print('=' * 60)

# Generar serie temporal sintética (ventas mensuales con tendencia y estacionalidad)
np.random.seed(42)
n_points = 200
time = np.arange(n_points)
trend = 0.05 * time
seasonal = 10 * np.sin(2 * np.pi * time / 12)
noise = np.random.normal(0, 2, n_points)
sales = 50 + trend + seasonal + noise

print(f'Serie temporal: {n_points} puntos')
print(f'Rango: {sales.min():.2f} - {sales.max():.2f}')

# Visualizar
plt.figure(figsize=(12, 4))
plt.plot(time, sales, 'b-', linewidth=2)
plt.title('Serie temporal: Ventas mensuales')
plt.xlabel('Mes')
plt.ylabel('Ventas')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Preparar datos para LSTM
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

seq_length = 12  # Usar 12 meses para predecir el siguiente

# Normalizar
scaler_lstm = StandardScaler()
sales_scaled = scaler_lstm.fit_transform(sales.reshape(-1, 1)).flatten()

# Crear secuencias
X_seq, y_seq = create_sequences(sales_scaled, seq_length)
X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)

# Split
split = int(0.8 * len(X_seq))
X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
y_train_seq, y_test_seq = y_seq[:split], y_seq[split:]

print(f'Secuencias creadas: {len(X_seq)}')
print(f'X_train: {X_train_seq.shape}')
print(f'X_test: {X_test_seq.shape}')

In [ ]:
# Modelo LSTM
model_lstm = models.Sequential([
    layers.LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
    layers.Dropout(0.2),
    layers.LSTM(50, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(25, activation='relu'),
    layers.Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.summary()

# Entrenar
print('\nEntrenando LSTM...')
history_lstm = model_lstm.fit(
    X_train_seq, y_train_seq,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# Predecir
y_pred_seq = model_lstm.predict(X_test_seq)

# Inverse transform
y_test_real = scaler_lstm.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
y_pred_real = scaler_lstm.inverse_transform(y_pred_seq).flatten()

# Métricas
mse = np.mean((y_test_real - y_pred_real) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test_real - y_pred_real))

print(f'\nMétricas:')
print(f'  MSE: {mse:.4f}')
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE: {mae:.4f}')

In [ ]:
# Visualizar predicciones
plt.figure(figsize=(12, 5))
plt.plot(y_test_real, 'b-', label='Real', linewidth=2)
plt.plot(y_pred_real, 'r--', label='Predicción LSTM', linewidth=2)
plt.title('LSTM: Predicción de series temporales')
plt.xlabel('Tiempo')
plt.ylabel('Ventas')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('\n' + '=' * 60)
print('CONCLUSIÓN: LSTMs son ideales para secuencias complejas.')
print('Pero para series simples, Prophet o ARIMA son más prácticos.')
print('=' * 60)

## Celda 6: Ejemplo con Transformers (sentiment analysis)

Los Transformers son el estado del arte para NLP.

**Metáfora:** Los Transformers son como "Google Maps del lenguaje": pueden ver todas las palabras a la vez y decidir cuáles son relevantes.

In [ ]:
# ============================================
# Transformers para análisis de sentimiento
# ============================================
print('=' * 60)
print('Transformers para análisis de sentimiento')
print('=' * 60)

# Cargar dataset de reseñas
try:
    df_reviews = pd.read_csv('../datos/datos_texto_reviews.csv')
    print(f'Reseñas cargadas: {len(df_reviews)}')
    print(f'\nDistribución de sentimiento:')
    print(df_reviews['sentiment'].value_counts())
    print(f'\nEjemplos:')
    print(df_reviews.head(10)[['text', 'sentiment', 'rating']])
except FileNotFoundError:
    print('Dataset no encontrado. Creando datos de ejemplo...')
    reviews_sample = [
        ('Excelente producto, superó mis expectativas.', 'positive'),
        ('Muy decepcionado con la compra.', 'negative'),
        ('Producto regular, nada especial.', 'neutral'),
        ('Lo recomiendo ampliamente.', 'positive'),
        ('Pésima calidad, no lo compren.', 'negative'),
    ]
    df_reviews = pd.DataFrame(reviews_sample, columns=['text', 'sentiment'])

In [ ]:
# Usar modelo pre-entrenado de Transformers
print('\nCargando modelo de sentiment analysis...')
try:
    classifier = pipeline(
        'sentiment-analysis',
        model='nlptown/bert-base-multilingual-uncased-sentiment',
        tokenizer='nlptown/bert-base-multilingual-uncased-sentiment'
    )
    print('Modelo cargado correctamente.')
    
    # Probar con reseñas de ejemplo
    test_reviews = [
        'Excelente producto, superó mis expectativas. La calidad es increíble.',
        'Muy decepcionado. El producto llegó dañado y la atención al cliente fue pésima.',
        'Regular, nada especial. Cumple con lo básico pero no destaca.',
        '¡Las mejores zapatillas que he tenido! Cómodas y con un diseño espectacular.',
        'No lo recomiendo. Se rompió después de una semana de uso.',
    ]
    
    print('\nPredicciones:')
    for review in test_reviews:
        result = classifier(review[:512])  # BERT tiene límite de 512 tokens
        print(f'\n  Texto: "{review[:60]}..."')
        print(f'  Sentimiento: {result[0]["label"]} (confianza: {result[0]["score"]:.4f})')
    
except Exception as e:
    print(f'Error al cargar modelo: {e}')
    print('Nota: Transformers requiere GPU para mejor rendimiento.')

In [ ]:
# Comparación: BERT vs enfoque tradicional (TF-IDF + Logistic Regression)
print('\n' + '=' * 60)
print('COMPARACIÓN: Transformers vs ML tradicional')
print('=' * 60)

comparison = {
    'BERT (Transformers)': {
        'Accuracy esperado': '~92-95%',
        'Tiempo entrenamiento': 'Horas (con GPU)',
        'Tiempo inferencia': '~50ms por texto',
        'Memoria': '~440MB',
        'Explicabilidad': 'Baja (caja negra)',
    },
    'TF-IDF + LR': {
        'Accuracy esperado': '~85-90%',
        'Tiempo entrenamiento': 'Minutos',
        'Tiempo inferencia': '~1ms por texto',
        'Memoria': '~10MB',
        'Explicabilidad': 'Media (coeficientes)',
    }
}

for model, metrics in comparison.items():
    print(f'\n{model}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value}')

print('\n' + '=' * 60)
print('CONCLUSIÓN: Transformers son más precisos pero más pesados.')
print('Para producción con latencia baja, considera modelos más ligeros.')
print('=' * 60)

## Celda 7: Cuándo parar de usar DL

El árbol de decisión para seleccionar tu modelo:

In [ ]:
# ============================================
# GUÍA: Cuándo usar qué modelo
# ============================================
print('=' * 60)
print('GUÍA: ¿Cuándo usar Deep Learning?')
print('=' * 60)

guide = """
┌─────────────────────────────────────────────────────────────┐
│                  ÁRBOL DE DECISIÓN                          │
│                                                             │
│  ¿Tus datos son tabulares?                                  │
│  ├── SÍ ──→ ¿Tienes <10M filas?                            │
│  │          ├── SÍ ──→ ¿Necesitas explicabilidad?           │
│  │          │         ├── SÍ ──→ Random Forest o Regresión  │
│  │          │         └── NO ──→ XGBoost o LightGBM         │
│  │          └── NO ──→ Gradient Boosting distribuido        │
│  │                                                             │
│  └── NO ──→ ¿Son imágenes?                                  │
│            ├── SÍ ──→ CNN (ResNet, EfficientNet)            │
│            └── NO ──→ ¿Son texto?                           │
│                      ├── SÍ ──→ Transformers (BERT, GPT)    │
│                      └── NO ──→ ¿Son audio?                 │
│                                ├── SÍ ──→ RNN/Transformers  │
│                                └── NO ──→ Representa como   │
│                                           datos tabulares    │
└─────────────────────────────────────────────────────────────┘
"""
print(guide)

print('\n' + '=' * 60)
print('RESUMEN EJECUTIVO:')
print('=' * 60)
print('''
1. Si tus datos son tabulares → NO uses Deep Learning
2. Si son imágenes, texto o audio → DL puede ser tu mejor opción
3. Siempre empieza con un baseline simple (regresión, árboles)
4. Si DL no supera significativamente al baseline, quédate con el baseline
5. La ética no es opcional —incrusta auditorías de sesgo en tu pipeline
''')

# Tabla comparativa final
print('\nTabla comparativa:')
print('-' * 80)
print(f'{"Escenario":<30} {"Mejor herramienta":<25} {"¿Por qué?":<25}')
print('-' * 80)
scenarios = [
    ('Predecir churn', 'XGBoost', 'Tabular, explicabilidad'),
    ('Clasificar emails', 'TF-IDF + LR', 'Rápido, suficiente'),
    ('Ventas mensuales', 'Prophet', 'Serie simple'),
    ('Radiografías', 'CNN', 'Imágenes'),
    ('Análisis sentimiento', 'BERT', 'Texto complejo'),
    ('Reconocimiento voz', 'LSTM/Transformer', 'Secuencias'),
]
for scenario, tool, reason in scenarios:
    print(f'{scenario:<30} {tool:<25} {reason:<25}')
print('-' * 80)

## Celda 8: Ética - sesgo en modelos de DL

Los modelos de DL son particularmente propensos a perpetuar y amplificar sesgos.

In [ ]:
# ============================================
# ÉTICA: Detección de sesgo en modelos de DL
# ============================================
print('=' * 60)
print('ÉTICA: Sesgo en modelos de Deep Learning')
print('=' * 60)

# Ejemplo conceptual de auditoría de sesgo
print("""
TIPOS DE SESGO EN DEEP LEARNING:

1. SESGO DE MUESTREO:
   - Datos no representativos de la población
   - Ejemplo: Modelo entrenado solo con fotos de personas blancas

2. SESGO DE MEDICIÓN:
   - Features que son proxies de raza/género
   - Ejemplo: Código postal como proxy de raza

3. SESGO DE ETIQUETADO:
   - Humanos que etiquetan con prejuicios
   - Ejemplo: Reclutadores que penalizan nombres "raros"

4. SESGO DE ALGORITMO:
   - El modelo amplifica sesgos sutiles
   - Ejemplo: Regresión que sobrestima salarios de hombres
""")

# Simular auditoría de fairness
np.random.seed(42)
n = 1000

# Simular predicciones con sesgo
gender = np.random.choice(['Male', 'Female'], n)
y_true = np.random.choice([0, 1], n, p=[0.7, 0.3])

# Simular predicciones con sesgo (hombres tienen mejor accuracy)
y_pred = y_true.copy()
noise_female = np.random.random(n) < 0.2  # 20% error para mujeres
noise_male = np.random.random(n) < 0.05  # 5% error para hombres

y_pred[(gender == 'Female') & noise_female] = 1 - y_pred[(gender == 'Female') & noise_female]
y_pred[(gender == 'Male') & noise_male] = 1 - y_pred[(gender == 'Male') & noise_male]

# Calcular métricas por grupo
print('\nEjemplo de auditoría de fairness:')
print('-' * 50)

for g in ['Male', 'Female']:
    mask = gender == g
    acc = accuracy_score(y_true[mask], y_pred[mask])
    n_samples = mask.sum()
    print(f'  {g}: Accuracy = {acc:.4f} (n={n_samples})')

# Disparate ratio
acc_male = accuracy_score(y_true[gender == 'Male'], y_pred[gender == 'Male'])
acc_female = accuracy_score(y_true[gender == 'Female'], y_pred[gender == 'Female'])
disparate_ratio = min(acc_male, acc_female) / max(acc_male, acc_female)

print(f'\n  Disparate Ratio: {disparate_ratio:.4f}')
if disparate_ratio < 0.8:
    print('  ⚠️  ADVERTENCIA: Posible discriminación detectada (< 0.8)')
else:
    print('  ✓  Fairness aceptable (>= 0.8)')

print('\n' + '=' * 60)
print('PREGUNTAS ÉTICAS OBLIGATORIAS:')
print('=' * 60)
print('''
Antes de部署 un modelo de DL que afecta vidas humanas:

1. PREGUNTA: "¿Quién puede ser perjudicado por este modelo?"
   → Identifica todos los grupos demográficos afectados

2. AUDITA: "¿El modelo funciona igual para todos los grupos?"
   → Calcula métricas por grupo (gender, race, age)

3. MONITOREA: "¿Qué pasa cuando los datos cambian?"
   → Implementa alertas de drift de fairness

4. EXPLICA: "¿Puedo justificar cada predicción ante un regulador?"
   → Documenta el modelo y sus limitaciones

Rajkomar et al. (2019): "La equidad y la explicabilidad deben ser
principios de diseño, no pensamientos posteriores."
''')

# Recursos
print('=' * 60)
print('RECURSOS PARA FAIRNESS:')
print('=' * 60)
print('''
- fairlearn: https://fairlearn.org/
- AI Fairness 360: https://aif360.mybluemix.net/
- What-If Tool: https://pair-code.github.io/what-if-tool/
- Google's Responsible AI: https://ai.google/responsibility/
''')